In [1]:
import pandas as pd

In [2]:
camera_inventory_df = pd.read_csv('../local_data/camera_inventory.tsv', delimiter="\t")
active_cameras_df = pd.read_csv('../local_data/active_cameras.tsv', delimiter="\t")
inactive_cameras_df = pd.read_csv('../local_data/inactive_cameras.tsv', delimiter="\t")

In [3]:
camera_inventory_df = camera_inventory_df.fillna('')
active_cameras_df = active_cameras_df.fillna('')
inactive_cameras_df = inactive_cameras_df.fillna('')

In [4]:
active_cameras_df = active_cameras_df.rename(columns={"Unnamed: 1": "Area"})
active_cameras_df = active_cameras_df.rename(columns={"Unnamed: 0": "TrapID"})
inactive_cameras_df = inactive_cameras_df.rename(columns={"Unnamed: 0": "TrapID"})

In [5]:
active_cameras_df.columns

Index(['TrapID', 'Area', 'County', 'Macro-Site', 'Micro-Site', 'Grid',
       'Latitude', 'Longitude', 'Elevation (ft)', 'Trail-type (dropdown)',
       'Habitat type', 'Secondary Habitat Type', 'Date Deployed',
       'Date Taken Down', 'Date Last Checked', 'Date to Be Checked',
       'Camera Status', 'Camera Model', 'Volunteer Contact', 'Python Lock #',
       'Padlock Type', 'Duplicate Python Key?', 'Contact before check?',
       'Send pictures?', 'Comments', 'Unnamed: 25', 'Unnamed: 26'],
      dtype='object')

In [6]:
inactive_cameras_df.columns

Index(['TrapID', 'Area', 'County', 'Macro-Site', 'Micro-Site', 'Grid',
       'Latitude', 'Longitude', 'Elevation (ft)', 'Trail-type (dropdown)',
       'Habitat type', 'Secondary Habitat Type', 'Date Deployed',
       'Date Taken Down', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16',
       'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20',
       'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23'],
      dtype='object')

## Validate the assumption of mutually-exclusion in the heirarchy

In [7]:
from collections import defaultdict

In [8]:
def show_overlaps(child, ancestor):
    ancestries = defaultdict(set)
    sheet_sources = {}
    for df, name in [(active_cameras_df, "active"), (inactive_cameras_df, "inactive")]:
        for _, record in df.iterrows():
            if record[child] and record[ancestor]:
                ancestries[record[child]].add(record[ancestor]) 
                sheet_sources[(record[child], record[ancestor])] = name
    print(f"====== {child} -> {ancestor} =====")
    for k in ancestries:
        if len(ancestries[k]) > 1:
            for v in ancestries[k]:
                print(f"{child} - {k}; {ancestor} - {v} (source - {sheet_sources[(k, v)]})")
            print("---")

### Check for different camera ids

In [9]:
child = "TrapID"
for ancestor in ["Micro-Site", "Grid", "Macro-Site", "County", "Area"]:
    show_overlaps(child, ancestor)    

====== TrapID -> Micro-Site =====
TrapID - COE10A; Micro-Site - first water trough location (source - inactive)
TrapID - COE10A; Micro-Site - Water Trough (source - active)
---
TrapID - SUP01A; Micro-Site - Near Sue P.s house (source - inactive)
TrapID - SUP01A; Micro-Site - Near Sue P.'s house (source - inactive)
---
====== TrapID -> Grid =====
====== TrapID -> Macro-Site =====
====== TrapID -> County =====
====== TrapID -> Area =====
TrapID - ROC01B; Area - East Bay (source - inactive)
TrapID - ROC01B; Area - North Bay (source - inactive)
---
TrapID - SUP01A; Area - South Bay (source - inactive)
TrapID - SUP01A; Area - North Bay (source - inactive)
---


### Check micro sites

In [10]:
child = "Micro-Site"
for ancestor in ["Grid", "Macro-Site", "County", "Area"]:
    show_overlaps(child, ancestor)    

====== Micro-Site -> Grid =====
Micro-Site - Montara; Grid - W. peninsula (source - inactive)
Micro-Site - Montara; Grid - SFPUC (source - inactive)
---
Micro-Site - Oat Hill Trail; Grid - MMWD (source - active)
Micro-Site - Oat Hill Trail; Grid - Calistoga (source - inactive)
---
====== Micro-Site -> Macro-Site =====
Micro-Site - Oat Hill Trail; Macro-Site - Robert Louis Stevenson State Park (source - inactive)
Micro-Site - Oat Hill Trail; Macro-Site - Marin Municipal Water District (source - active)
---
Micro-Site - Bolinas Ridge; Macro-Site - Golden Gate National Rec Area (source - active)
Micro-Site - Bolinas Ridge; Macro-Site - Marin Municipal Water District (source - inactive)
---
Micro-Site - Rancho Corral de Tierra; Macro-Site - Rancho Corral de Tierra (source - inactive)
Micro-Site - Rancho Corral de Tierra; Macro-Site - Rancho Corral de Tierra (NPS) (source - active)
---
Micro-Site - Skyline Trail; Macro-Site - Wunderlich County Park (source - inactive)
Micro-Site - Skyline T

### Check Grid

In [13]:
# child = "Grid"
# for ancestor in ["Macro-Site", "County", "Area"]:
#     show_overlaps(child, ancestor)    

====== Grid -> Macro-Site =====
Grid - woodside; Macro-Site - MidPeninsula Open Space Preserve (source - active)
Grid - woodside; Macro-Site - Huddart County Park (SMCP) (source - active)
Grid - woodside; Macro-Site - Edgewood County Park (source - inactive)
Grid - woodside; Macro-Site - Phleger Estate (NPS) (source - active)
Grid - woodside; Macro-Site - Woodside (source - inactive)
Grid - woodside; Macro-Site - Skyline Blvd (source - inactive)
Grid - woodside; Macro-Site - Filoli Estate (source - active)
Grid - woodside; Macro-Site - Wunderlich County Park (source - inactive)
Grid - woodside; Macro-Site - Wunderlich County Park (SMCP) (source - active)
Grid - woodside; Macro-Site - Edgewood County Park (SMCP) (source - inactive)
---
Grid - W. peninsula; Macro-Site - San Pedro Valley County Park (SMCP) (source - active)
Grid - W. peninsula; Macro-Site - San Pedro Valley County Park (source - inactive)
Grid - W. peninsula; Macro-Site - Rancho Corral de Tierra (NPS) (source - active)
Gr

### Check macro sites

In [11]:
child = "Macro-Site"
for ancestor in ["County", "Area"]:
    show_overlaps(child, ancestor)    

====== Macro-Site -> County =====
Macro-Site - La Honda; County - San Mateo (source - inactive)
Macro-Site - La Honda; County - san mateo (source - active)
---
Macro-Site - SFPUC; County - San Mateo (source - inactive)
Macro-Site - SFPUC; County -  San Mateo (source - inactive)
---
Macro-Site - Marin Municipal Water District; County - Marin (source - inactive)
Macro-Site - Marin Municipal Water District; County - Napa (source - inactive)
---
====== Macro-Site -> Area =====
Macro-Site - Rancho Corral de Tierra; Area - South Bay (source - inactive)
Macro-Site - Rancho Corral de Tierra; Area - North Bay (source - inactive)
---
Macro-Site - Solano Land Trust / Rockville Trails; Area - East Bay (source - inactive)
Macro-Site - Solano Land Trust / Rockville Trails; Area - North Bay (source - inactive)
---


### Check County

In [12]:
child = "County"
for ancestor in ["Area"]:
    show_overlaps(child, ancestor)    

====== County -> Area =====
County - San Mateo; Area - Peninsula (source - active)
County - San Mateo; Area - South Bay (source - inactive)
County - San Mateo; Area - North Bay (source - inactive)
---
County - Contra Costa; Area - East Bay-Upper (source - inactive)
County - Contra Costa; Area - East Bay (source - inactive)
---
County - Santa Clara; Area - East Bay-Lower (source - inactive)
County - Santa Clara; Area - South Bay (source - inactive)
---
County - Solano; Area - East Bay (source - inactive)
County - Solano; Area - North Bay (source - inactive)
---
